# Fine Tuning Gemma 3

In [1]:
import os

is_colab = False
is_sagemaker = False
device = "mps"

env_keys = os.environ.keys()

if any(k.startswith("COLAB_") for k in env_keys):
    is_colab = True
    device = "cuda"
    print("Running in Google Colab")

elif "SM_CURRENT_HOST" in os.environ:
    is_sagemaker = True
    device = "cuda"
    print("Running in SageMaker")

else:
    print("Running locally")


Running locally


In [2]:
device

'mps'

In [3]:
if is_colab or is_sagemaker:
    !nvidia-smi

In [4]:
%%capture
if is_colab:
    # !pip install -U bitsandbytes transformers datasets peft acecelerate
    !pip install -U transformers==4.57.3 evaluate

In [5]:
if is_colab:
    from google.colab import userdata
    # os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    # from transformers.utils.quantization_config import BitsAndBytesConfig
    # quantization_config = BitsAndBytesConfig(load_in_8bit=True)
# os.environ["HF_TOKEN"] = ""

In [6]:
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from peft import PeftModel, PeftConfig
from transformers.trainer import Trainer
from transformers.training_args import TrainingArguments
from transformers.data.data_collator import DataCollatorForLanguageModeling
from peft import prepare_model_for_kbit_training
from tqdm.auto import tqdm
from transformers.optimization import get_cosine_schedule_with_warmup
from datasets import load_from_disk

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

## Loading the Model

In [34]:
from transformers import AutoModelForCausalLM, AutoTokenizer
base_model_id = "google/gemma-3-270m-it"

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    attn_implementation="eager",
    # quantization_config=quantization_config if is_colab else None,
)

tokenizer.padding_side = "right"

In [9]:
for name, param in model.named_parameters():
    print(name, param.device)
    break

model.embed_tokens.weight mps:0


In [10]:
print(tokenizer.get_chat_template())

{{ bos_token }}
{%- if messages[0]['role'] == 'system' -%}
    {%- if messages[0]['content'] is string -%}
        {%- set first_user_prefix = messages[0]['content'] + '

' -%}
    {%- else -%}
        {%- set first_user_prefix = messages[0]['content'][0]['text'] + '

' -%}
    {%- endif -%}
    {%- set loop_messages = messages[1:] -%}
{%- else -%}
    {%- set first_user_prefix = "" -%}
    {%- set loop_messages = messages -%}
{%- endif -%}
{%- for message in loop_messages -%}
    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) -%}
        {{ raise_exception("Conversation roles must alternate user/assistant/user/assistant/...") }}
    {%- endif -%}
    {%- if (message['role'] == 'assistant') -%}
        {%- set role = "model" -%}
    {%- else -%}
        {%- set role = message['role'] -%}
    {%- endif -%}
    {{ '<start_of_turn>' + role + '
' + (first_user_prefix if loop.first else "") }}
    {%- if message['content'] is string -%}
        {{ message['content'] | trim }}


**Testing the prompt before fine-tuning:**

In [ ]:
messages = [
    {"role": "system", "content": "You are a assistant responsible for classifying mental health status."},
    {"role": "user", "content": "I am depressed and want to die"}
]


input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(device)
attention_mask = torch.ones_like(input_ids).to(device)

outputs = model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_new_tokens=100,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id # Using eos_token_id as pad_token_id for Gemma
)

input_len = input_ids.shape[1]
generated_tokens_tensor = outputs[0, input_len:]
decoded_response = tokenizer.decode(generated_tokens_tensor, skip_special_tokens=True)

print(decoded_response)

I understand. It's very difficult to say what's happening, but I can offer some resources and support to help you cope. Please reach out to a mental health professional or a crisis hotline. You can also contact the National Suicide Prevention Services at 988 and the Crisis Text Line at 741740 in the US.



In [12]:
print(tokenizer.apply_chat_template(messages,tokenize = False, add_generation_prompt=True))

<bos><start_of_turn>user
You are a assistant responsible for classifying mental health status.

I am depressed and want to die<end_of_turn>
<start_of_turn>model



## Data Setup

In [11]:
ds = load_dataset("nbertagnolli/counsel-chat")

# drop null
ds = ds.filter(lambda x: x['questionText'] is not None)
ds = ds.filter(lambda x: x['topic'] is not None)

Repo card metadata block was not found. Setting CardData to empty.


Filter:   0%|          | 0/2775 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2636 [00:00<?, ? examples/s]

In [8]:
ds

DatasetDict({
    train: Dataset({
        features: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views'],
        num_rows: 2636
    })
})

In [9]:
ds["train"][0]

{'questionID': 0,
 'questionTitle': 'Do I have too many issues for counseling?',
 'questionText': 'I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.\n   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?',
 'questionLink': 'https://counselchat.com/questions/do-i-have-too-many-issues-for-counseling',
 'topic': 'depression',
 'therapistInfo': 'Jennifer MolinariHypnotherapist & Licensed Counselor',
 'therapistURL': 'https://counselchat.com/therapists/jennifer-molinari',
 'answerText': 'It is very common for\xa0people to have multiple issues that they want to (and need to) address in counseling.\xa0 I have had clients ask that same question and through more exploration, there is often an underlying fear that they\xa0 "can\

In [12]:
SYSTEM_PROMPT = "You are an assistant responsible for classifying mental health status."

def build_chat(batch):
    questions = batch["questionText"]
    topics = batch["topic"]

    batch_messages = []
    batch_responses = []
    batch_conversations = []

    for q, t in zip(questions, topics):
        batch_messages.append([
            {"role": "user", "content": SYSTEM_PROMPT + "\n" + q},
        ])

        batch_responses.append(f"Based on what you've described, this sounds like '{t}'.")

        batch_conversations.append([
            {"role": "user", "content": SYSTEM_PROMPT + "\n" + q},
            {"role": "assistant", "content": f"Based on what you've described, this sounds like '{t}'."}
        ])

    return {
        "messages": batch_messages,
        "response": batch_responses,
        "conversation": batch_conversations
    }

chat_dataset = ds.map(build_chat, batched=True)

Map:   0%|          | 0/2636 [00:00<?, ? examples/s]

In [11]:
chat_dataset

DatasetDict({
    train: Dataset({
        features: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views', 'messages', 'response', 'conversation'],
        num_rows: 2636
    })
})

In [13]:
MAX_LENGTH = 512
IGNORE_INDEX = -100


def format_conversations(batch):
    """
    Applies the model's chat template to a list of conversational turns.
    """
    texts = []
    for conversation in batch['conversation']:
        formatted_text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False 
        ).removeprefix('<bos>')
        texts.append(formatted_text)
    return {"text": texts}

formatted_dataset = chat_dataset.map(format_conversations, batched=True)

Parameter 'function'=<function format_conversations at 0x349e5e840> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/2636 [00:00<?, ? examples/s]

In [13]:
formatted_dataset

DatasetDict({
    train: Dataset({
        features: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views', 'messages', 'response', 'conversation', 'text'],
        num_rows: 2636
    })
})

In [14]:
formatted_dataset.column_names

{'train': ['questionID',
  'questionTitle',
  'questionText',
  'questionLink',
  'topic',
  'therapistInfo',
  'therapistURL',
  'answerText',
  'upvotes',
  'views',
  'messages',
  'response',
  'conversation',
  'text']}

In [15]:
print(formatted_dataset["train"][0]["text"])

<start_of_turn>user
You are an assistant responsible for classifying mental health status.
I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.
   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?<end_of_turn>
<start_of_turn>model
Based on what you've described, this sounds like 'depression'.<end_of_turn>



In [16]:
formatted_dataset["train"][0]["conversation"]

[{'content': 'You are an assistant responsible for classifying mental health status.\nI have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.\n   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?',
  'role': 'user'},
 {'content': "Based on what you've described, this sounds like 'depression'.",
  'role': 'assistant'}]

In [14]:
def tokenize_and_mask_labels(batch):
    """
    Tokenizes the text and creates labels, masking the instruction/user tokens.
    """
    tokenized_results = []

    for conversation in batch['conversation']:
        full_text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False
        ).removeprefix('<bos>')


        prompt_conversation = conversation[:-1]

        prompt_text = tokenizer.apply_chat_template(
            prompt_conversation,
            tokenize=False,
            add_generation_prompt=True # Ensures the model's response start token is included
        ).removeprefix('<bos>')

        full_tokenized = tokenizer(
            full_text,
            max_length=MAX_LENGTH,
            truncation=True,
            padding=False,
            return_tensors=None,
        )

        prompt_tokenized = tokenizer(
            prompt_text,
            max_length=MAX_LENGTH,
            truncation=True,
            padding=False,
            return_tensors=None,
        )

        labels = full_tokenized["input_ids"].copy()

        prompt_length = len(prompt_tokenized["input_ids"])

        mask_end = min(prompt_length, len(labels))
        labels[:mask_end] = [IGNORE_INDEX] * mask_end

        input_ids = full_tokenized["input_ids"][:-1]
        labels = labels[1:]
        attention_mask = full_tokenized["attention_mask"][:-1]

        tokenized_results.append({
            "input_ids": input_ids,
            "labels": labels,
            "attention_mask": attention_mask,
            "prompt_input_ids": prompt_tokenized["input_ids"],
            "prompt_attention_mask": prompt_tokenized["attention_mask"]
        })

    return {
        "input_ids": [r["input_ids"] for r in tokenized_results],
        "labels": [r["labels"] for r in tokenized_results],
        "attention_mask": [r["attention_mask"] for r in tokenized_results],
        "prompt_input_ids": [r["prompt_input_ids"] for r in tokenized_results],
        "prompt_attention_mask": [r["prompt_attention_mask"] for r in tokenized_results]
    }



In [15]:
tokenizer.pad_token, tokenizer.eos_token

('<pad>', '<eos>')

In [16]:
tokenized_dataset = formatted_dataset.map(
    tokenize_and_mask_labels,
    batched=True,
    remove_columns=formatted_dataset["train"].column_names
)

Map:   0%|          | 0/2636 [00:00<?, ? examples/s]

In [20]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'attention_mask', 'prompt_input_ids', 'prompt_attention_mask'],
        num_rows: 2636
    })
})

In [ ]:
sample = tokenized_dataset['train'][3]


print("\n--- Processed Sample (Input IDs and Labels) ---")
print("Input IDs (first 20):", sample["input_ids"][:20])
print("Labels (first 20):  ", sample["labels"][:20])

print("Input IDs (first 20):", sample["input_ids"])
print("Labels (first 20):  ", sample["labels"])

decoded_input = tokenizer.decode(sample["input_ids"])
print("\nDecoded Input (Full Context):")
print(decoded_input)

mask_start = next(i for i, label in enumerate(sample['labels']) if label != IGNORE_INDEX)
print(f"\nMasking Check:")
print(f"Index of first non-{IGNORE_INDEX} label (start of response): {mask_start}")

first_response_token = sample["labels"][mask_start]
decoded_response_start = tokenizer.decode(first_response_token)

print(f"Decoded token at response start: '{decoded_response_start}'")

is_prompt_masked = all(label == IGNORE_INDEX for label in sample["labels"][:mask_start])
print(f"Are all prompt labels masked (-100)? {is_prompt_masked}")


print("\n--- Dataset ready for Hugging Face Trainer ---")


--- Processed Sample (Input IDs and Labels) ---
Input IDs (first 20): [2, 105, 2364, 107, 3048, 659, 614, 16326, 7757, 573, 99896, 9069, 2404, 4981, 236761, 107, 236777, 735, 834, 1551]
Labels (first 20):   [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
Input IDs (first 20): [2, 105, 2364, 107, 3048, 659, 614, 16326, 7757, 573, 99896, 9069, 2404, 4981, 236761, 107, 236777, 735, 834, 1551, 4342, 531, 3421, 236761, 564, 735, 496, 4083, 529, 11953, 16407, 236764, 564, 236858, 236757, 496, 16489, 7923, 72399, 532, 564, 1006, 496, 19418, 1728, 542, 218518, 236761, 140, 236777, 735, 496, 1440, 4083, 529, 17998, 532, 564, 236858, 236757, 6534, 531, 735, 17660, 236761, 564, 735, 2708, 1265, 78114, 840, 564, 236858, 560, 1010, 38968, 11578, 573, 4180, 236743, 236800, 236810, 1518, 236761, 107, 139, 236777, 236858, 560, 2752, 1053, 45899, 1003, 1027, 529, 672, 236761, 3574, 564, 735, 2311, 1551, 4342, 531, 3421, 528, 4589

In [23]:
tokenizer.padding_side, tokenizer.truncation_side

('right', 'right')

In [21]:
tokenized_dataset.save_to_disk("../data/processed/tokenized_counsel_chat_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/2636 [00:00<?, ? examples/s]

In [17]:
tokenized_dataset = load_from_disk("../data/processed/tokenized_counsel_chat_dataset")
# tokenized_dataset.load_from_disk("../data/processed/tokenized_counsel_chat_dataset")

In [18]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'attention_mask', 'prompt_input_ids', 'prompt_attention_mask'],
        num_rows: 2636
    })
})

In [19]:
def validate_causal_shift(dataset, tokenizer=None, num_samples=5):
    """
    Checks whether labels are correctly shifted for causal LM.
    """
    import random
    import torch

    indices = random.sample(range(len(dataset)), num_samples)

    for idx in indices:
        sample = dataset[idx]
        input_ids = torch.tensor(sample["input_ids"])
        labels = torch.tensor(sample["labels"])

        valid = labels != -100

        shifted_input = input_ids[1:][valid[:-1]]
        shifted_labels = labels[:-1][valid[:-1]]

        is_shifted = torch.equal(shifted_input, shifted_labels)

        print(f"\nSample {idx}: shifted = {is_shifted}")

        if tokenizer:
            print("Input :", tokenizer.decode(input_ids))
            print("Labels:", tokenizer.decode(labels[labels != -100]))

        if not is_shifted:
            print("Mismatch detected")
        else:
            print("Correctly shifted")


In [20]:
from torch.nn.utils.rnn import pad_sequence

def causal_lm_collator(batch):
    input_ids = [x["input_ids"] for x in batch]
    labels = [x["labels"] for x in batch]
    # prompt_input_ids = [x["prompt_input_ids"] for x in batch]
    # prompt_attention_mask = [x["prompt_attention_mask"] for x in batch]

    input_ids = pad_sequence(
        input_ids, batch_first=True, padding_value=tokenizer.pad_token_id
    )

    labels = pad_sequence(
        labels, batch_first=True, padding_value=-100
    )

    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
        # "prompt_input_ids": prompt_input_ids,
        # "prompt_attention_mask": prompt_attention_mask,
    }

from torch.utils.data import Dataset, DataLoader

class LMDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "prompt_input_ids": torch.tensor(item["prompt_input_ids"], dtype=torch.long),
            "prompt_attention_mask": torch.tensor(item["prompt_attention_mask"], dtype=torch.long),
        }

In [22]:
split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.97, seed=42)
split_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'attention_mask', 'prompt_input_ids', 'prompt_attention_mask'],
        num_rows: 79
    })
    test: Dataset({
        features: ['input_ids', 'labels', 'attention_mask', 'prompt_input_ids', 'prompt_attention_mask'],
        num_rows: 2557
    })
})

In [23]:
validate_causal_shift(split_dataset["train"], tokenizer)


Sample 67: shifted = True
Input : <bos><start_of_turn>user
You are an assistant responsible for classifying mental health status.
I keep being mean to my best friend, and I don't know why all the time. I did come to maybe some kind of conclusion that it is because my mother is mean to me all the time. Could that be a cause?<end_of_turn>
<start_of_turn>model
Based on what you've described, this sounds like 'family-conflict'.<end_of_turn>
Labels: Based on what you've described, this sounds like 'family-conflict'.<end_of_turn>

Correctly shifted

Sample 55: shifted = True
Input : <bos><start_of_turn>user
You are an assistant responsible for classifying mental health status.
Now that the other girl is out of the picture, our sex life isn't the same. Is it because he is still thinking about the other girl?<end_of_turn>
<start_of_turn>model
Based on what you've described, this sounds like 'relationships'.<end_of_turn>
Labels: Based on what you've described, this sounds like 'relationships'.

In [24]:
batch_size=2

train_dataset = LMDataset(split_dataset["train"])

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=causal_lm_collator,
)

val_dataset = LMDataset(split_dataset["test"])
val_dataloader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    # collate_fn=causal_lm_collator,
)

In [25]:
for batch in train_dataloader:
    print(batch)
    break

{'input_ids': tensor([[     2,    105,   2364,    107,   3048,    659,    614,  16326,   7757,
            573,  99896,   9069,   2404,   4981, 236761,    107,   3910,    776,
          85736,  47680,    506,   1722,    529,  18237,    910,   7391,    735,
         236881,   2900,   9507,    776,    901,   1385,    657,    531,  10741,
           1144,   1722,    529,  18237,    506,   4430,    815, 236881,    564,
         236789,    560,   1676,   1003,    672,   3489, 236764,    840,    564,
         236789, 236753,   1133,    531,   3050,    672,    699,    496,  46063,
         236789, 236751,   1523,    529,   1927, 236761,    106,    107,    105,
           4368,    107,  22515,    580,   1144,    611, 236789,    560,   4970,
         236764,    672,  12054,   1133,    756, 134915,   6748,    106],
        [     2,    105,   2364,    107,   3048,    659,    614,  16326,   7757,
            573,  99896,   9069,   2404,   4981, 236761,    107, 236777, 236789,
         236757,    4

In [26]:
print({k: v.shape for k, v in batch.items()})

{'input_ids': torch.Size([2, 89]), 'labels': torch.Size([2, 89]), 'attention_mask': torch.Size([2, 89])}


## PEFT Setup

In [27]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

**Orginal Model**

In [35]:
model

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps

In [26]:
# try:
#     peft_model = peft_model.unload()
# except Exception as e:
#     print("Error in get_peft_model:", e)
#     raise e

# peft_model = peft_model.unload()

In [36]:
# peft_model.unload()
train_model = prepare_model_for_kbit_training(model)
peft_model = get_peft_model(train_model, lora_config)

peft_model.enable_input_require_grads()
peft_model.gradient_checkpointing_enable()
peft_model.config.use_cache = False

peft_model.print_trainable_parameters();


trainable params: 737,280 || all params: 268,835,456 || trainable%: 0.2742


**Peft Model**

In [37]:
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForCausalLM(
      (model): Gemma3TextModel(
        (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x Gemma3DecoderLayer(
            (self_attn): Gemma3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=640, out_features=1024, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=640, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1024, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
             

## Training Loop

In [38]:
device

'mps'

In [29]:
total_steps = len(train_dataloader) * 3
total_steps

120

In [38]:
from torch.amp.grad_scaler import GradScaler
from torch.amp.autocast_mode import autocast


criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(
    peft_model.parameters(),
    lr=5e-4,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=50, # ~5% of total steps
    num_training_steps=total_steps
)

scaler = GradScaler()

In [39]:
# scheduler_steps = []
# for step in range(total_steps):
#     scheduler_steps.append(scheduler.get_last_lr()[0])
#     scheduler.step()

In [40]:
# import matplotlib.pyplot as plt
# plt.plot(scheduler_steps)
# plt.xlabel("Step")
# plt.ylabel("Learning Rate")
# plt.title("Learning Rate Schedule")
# plt.show()

In [41]:
num_epochs = 3
accum_steps = 4
num_training_steps = num_epochs * len(train_dataloader) // accum_steps

progress_bar = tqdm(range(num_training_steps))

peft_model.train()
optimizer.zero_grad()

for epoch in range(num_epochs):
    for step, batch in enumerate(train_dataloader):
        with autocast(device_type=device, dtype=torch.float16):
            outputs = peft_model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            )
            logits = outputs.logits
            labels = batch["labels"].to(device)
            loss = criterion(
                logits.view(-1, logits.size(-1)),
                labels.view(-1)
            )
            loss = loss / accum_steps
        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0:
            print(f"Epoch {epoch+1}, Step {step+1}, Loss: {loss.item() * accum_steps:.4f}")

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            progress_bar.update(1)

    if (step + 1) % accum_steps != 0:
        print(f"Epoch {epoch+1}, Step {step+1}, Loss: {loss.item() * accum_steps:.4f}")

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()
        progress_bar.update(1)


  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 1, Step 4, Loss: 4.8307
Epoch 1, Step 8, Loss: 6.4486
Epoch 1, Step 12, Loss: 4.5387
Epoch 1, Step 16, Loss: 4.3088
Epoch 1, Step 20, Loss: 4.8591
Epoch 1, Step 24, Loss: 4.6052
Epoch 1, Step 28, Loss: 4.6131
Epoch 1, Step 32, Loss: 5.0515
Epoch 1, Step 36, Loss: 4.0484
Epoch 1, Step 40, Loss: 5.0918
Epoch 2, Step 4, Loss: 3.1672
Epoch 2, Step 8, Loss: 2.3687
Epoch 2, Step 12, Loss: 2.4238
Epoch 2, Step 16, Loss: 2.1409
Epoch 2, Step 20, Loss: 2.0586
Epoch 2, Step 24, Loss: 2.7204
Epoch 2, Step 28, Loss: 1.9340
Epoch 2, Step 32, Loss: 1.7805
Epoch 2, Step 36, Loss: 1.2993
Epoch 2, Step 40, Loss: 0.7760
Epoch 3, Step 4, Loss: 0.7981
Epoch 3, Step 8, Loss: 0.9281
Epoch 3, Step 12, Loss: 1.3277
Epoch 3, Step 16, Loss: 0.5632
Epoch 3, Step 20, Loss: 0.6554
Epoch 3, Step 24, Loss: 0.4837
Epoch 3, Step 28, Loss: 0.7971
Epoch 3, Step 32, Loss: 0.3917
Epoch 3, Step 36, Loss: 0.3560
Epoch 3, Step 40, Loss: 1.8812


## Inference After Fine-Tuning

In [ ]:
save_dir = "gemma-lora-adapter"

In [44]:
peft_model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

('gemma-lora-adapter_3/tokenizer_config.json',
 'gemma-lora-adapter_3/special_tokens_map.json',
 'gemma-lora-adapter_3/chat_template.jinja',
 'gemma-lora-adapter_3/tokenizer.model',
 'gemma-lora-adapter_3/added_tokens.json',
 'gemma-lora-adapter_3/tokenizer.json')

In [45]:
device

'mps'

In [46]:
base_model_id = "google/gemma-3-270m-it" 
# adapter_path = "gemma-lora-adapter"

tokenizer = AutoTokenizer.from_pretrained(save_dir)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float32,
    device_map="auto",
    attn_implementation="eager",
)

model = PeftModel.from_pretrained(base_model, save_dir)

In [47]:
print("Tokenizer vocab size:", len(tokenizer))
print("Model embedding size:", model.get_input_embeddings().weight.shape[0])

Tokenizer vocab size: 262145
Model embedding size: 262144


In [48]:
if device == "mps":
    model.to("cpu")
model.resize_token_embeddings(len(tokenizer))
model.to(device)
model.eval();

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [49]:
print("Tokenizer vocab size:", len(tokenizer))
print("Model embedding size:", model.get_input_embeddings().weight.shape[0])

Tokenizer vocab size: 262145
Model embedding size: 262145


In [50]:
for name, param in model.named_parameters():
    print(name, param.device)
    break

base_model.model.model.embed_tokens.weight mps:0


In [56]:
messages = [
    {"role": "user", "content": "You are a assistant responsible for classifying mental health status. I am bored and sad"}
]

input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True,).to(device)
attention_mask = torch.ones_like(input_ids).to(device)

In [57]:
with torch.no_grad():
    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=100,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.1,
    )

input_len = input_ids.shape[1]
generated_tokens_tensor = outputs[0, input_len:]
decoded_response = tokenizer.decode(generated_tokens_tensor, skip_special_tokens=True)

print(decoded_response)

Based on what you've described, this sounds like 'depression'.


In [58]:
model.eval()
predictions = []
references = []
samples = 10

for batch in val_dataloader:
    input_ids = batch["prompt_input_ids"].to(device)
    attention_mask = batch["prompt_attention_mask"].to(device)
    labels = batch["labels"].to(device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=20,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
             temperature=0.1,
        )


    pred = outputs[0, input_ids.shape[1]:]

    decoded_pred = tokenizer.batch_decode(
        pred,
        skip_special_tokens=True,
    )

    labels_decoded = tokenizer.batch_decode(
        labels[0][labels[0] != -100],
        skip_special_tokens=True,
    )

    predictions.append("".join(decoded_pred))
    references.append("".join(labels_decoded))

    
    if samples <= 0:
        break
    samples -= 1
    print();

In [59]:
predictions

["Based on what you've described, this sounds like 'depression'.",
 "Based on what you've described, this sounds like 'emotional distress'.",
 "Based on what you've described, this sounds like 'emotional intelligence'.",
 "Based on what you've described, this sounds like 'sex'.",
 "Based on what you've described, this sounds like 'depression'.",
 "Based on what you've described, this sounds like 'fear'.",
 "Based on what you've described, this sounds like 'friendship'.",
 "Based on what you've described, this sounds like 'therapy'.",
 "Based on what you've described, this sounds like 'therapy'.",
 "Based on what you've described, this sounds like 'romantic'.",
 "Based on what you've described, this sounds like 'stress'."]

In [60]:
references

["Based on what you've described, this sounds like 'anxiety'.\n",
 "Based on what you've described, this sounds like 'anxiety'.\n",
 "Based on what you've described, this sounds like 'marriage'.\n",
 "Based on what you've described, this sounds like 'intimacy'.\n",
 "Based on what you've described, this sounds like 'anxiety'.\n",
 "Based on what you've described, this sounds like 'behavioral-change'.\n",
 "Based on what you've described, this sounds like 'relationships'.\n",
 "Based on what you've described, this sounds like 'parenting'.\n",
 "Based on what you've described, this sounds like 'counseling-fundamentals'.\n",
 "Based on what you've described, this sounds like 'intimacy'.\n",
 "Based on what you've described, this sounds like 'anxiety'.\n"]

In [40]:
input_ids.shape, attention_mask.shape, labels.shape, outputs.shape

(torch.Size([1, 64]),
 torch.Size([1, 64]),
 torch.Size([1, 79]),
 torch.Size([1, 79]))

## Merging the Model

In [41]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained("./gemma-merged")
tokenizer.save_pretrained("./gemma-merged");

In [42]:
tuned_model = AutoModelForCausalLM.from_pretrained("./gemma-merged", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("./gemma-merged")

# chatbot = pipeline("text-generation", model=tuned_model, tokenizer=tokenizer, device=0);

In [ ]:
messages = [
    {"role": "user", "content": "You are a assistant responsible for classifying mental health status. I feel anxious and stressed all the time."}
]


input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(device)
attention_mask = torch.ones_like(input_ids).to(device)

outputs = tuned_model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_new_tokens=100,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id 
    temparature=0.1,
)

input_len = input_ids.shape[1]
generated_tokens_tensor = outputs[0, input_len:]
decoded_response = tokenizer.decode(generated_tokens_tensor, skip_special_tokens=True)

print(decoded_response)

Based on what you've described, this sounds like 'stress'.


In [ ]:
# training_args = TrainingArguments(
#     output_dir="./gemma-finetuned-model",
#     per_device_train_batch_size=4,
#     num_train_epochs=3,
#     logging_dir='./logs',
#     # save_steps=500,
#     logging_steps=100,
#     save_strategy="no",
#     label_names=["labels"],  # Explicitly specify label names for PEFT models
#     save_total_limit=1,
#     report_to="none",
#     learning_rate=0.0001,
# )

# data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# trainer = Trainer(
#     model=peft_model,
#     args=training_args,
#     train_dataset=tokenized_dataset["train"],
#     processing_class=tokenizer,
#     data_collator=data_collator,
# );

# trainer.train()
# trainer.save_model("./finetuned-model-gemma")